# Setup

In [1]:
%%capture
!pip install google-api-python-client>=2.0
!pip install google-auth>=2.0
!pip install google-auth-httplib2>=0.2
!pip install google-auth-oauthlib>=1.0
!pip install PytorchWildlife

In [4]:
from pathlib import Path
from typing import Any, Iterator
import json
import shutil
import sys
import time
import torch
from tqdm.auto import tqdm
from PytorchWildlife.models import detection as pw_detection

from google_drive_client import GoogleDriveClient, DriveClientInitArgs


In [5]:
gdrive = GoogleDriveClient(
    DriveClientInitArgs(
        client_name="google_api",
        credentials_file="echo-saviia-credentials.json",
        root_folder_id="1LVUSQJjcuPEDt4taa_xgizsOih6vRTTb",
    )
)

In [6]:
ORIGINAL_MAP_PATH = Path("data/dirs.json")
PREPROCESSED_MAP_PATH = Path("data/dirs_preprocessed.json")
OUTPUT_PATH = Path("data/wild_detector_results.json")
TEMP_PATH = Path(".tmp/wild_detector")

DOWNLOAD_BATCH_SIZE = 16
DOWNLOAD_WORKERS = 6
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}

TEMP_PATH.mkdir(parents=True, exist_ok=True)


# Helpers


In [7]:
def load_json(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("r") as file:
        return json.load(file)


def save_json(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )
    temporary_path.replace(path)


def chunks(values: list, size: int) -> Iterator[list]:
    for index in range(0, len(values), size):
        yield values[index:index + size]


def to_serializable(value: Any) -> list:
    if hasattr(value, "detach"):
        value = value.detach().cpu()
    if hasattr(value, "tolist"):
        return value.tolist()
    return list(value)

In [8]:
def extract_keys_path(name, map, keys):
    keys.append(name)
    if isinstance(map[name], list):
        return keys
    for key in map[name]:
        result = extract_keys_path(key, map[name], keys)
        if result:
            return result
    keys.pop()
    return None


def extract_last_value(map: dict, keys: list[str]):
    curr_map = map
    for key in keys:
        curr_map = curr_map[key]
    return [val for val in curr_map if not val.startswith("._")]

def check_routes(fpath: str, name: str) -> dict:
    path = str(Path(fpath) / name)
    entries = list(gdrive.list_dir(path))
    if any(f.entry_type == "file" for f in entries):
        return {name: [f.name for f in entries if f.entry_type == "file"]}
    dirs = [f.name for f in entries if f.entry_type == "directory"]
    children = {}
    for cname in dirs:
        children.update(check_routes(path, cname))
    return {name: children}



# Load directories information

In [ ]:
DOWNLOAD_PATH_MAP = False # For the first time, both parameters must be true. 
PREPROCESS_PATH_MAP = False
SOURCE = "Data"
START_DIR = "Sin clasificar"


if DOWNLOAD_PATH_MAP:
    map = check_routes(SOURCE, START_DIR)
    result = {}
    result[SOURCE] = map
    save_json(ORIGINAL_MAP_PATH, result)
else:
    map = load_json(ORIGINAL_MAP_PATH)[SOURCE][START_DIR]
if PREPROCESS_PATH_MAP:
    preprocessed_map = {}
    for key in map.keys():
        keys = extract_keys_path(key, map, [])
        values = extract_last_value(map, keys)  # type: ignore
        dirpath = f"{SOURCE}/{START_DIR}/" + "/".join(keys)  # type: ignore
        preprocessed_map[dirpath] = values
    save_json(PREPROCESSED_MAP_PATH, preprocessed_map)
else:
    preprocessed_map = load_json(PREPROCESSED_MAP_PATH)

# Model detector

In [16]:
# Load the detector model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
detector = pw_detection.MegaDetectorV6(device=DEVICE, version="MDV6-yolov10-e")
results: dict[str, list[dict]] = load_json(OUTPUT_PATH)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv10x summary (fused): 191 layers, 29,399,417 parameters, 0 gradients, 160.0 GFLOPs


# Detection

In [ ]:
for folder_path, expected_names in preprocessed_map.items():
    print(f"\nProcessing: {folder_path}")

    folder_results = results.setdefault(folder_path, [])
    already_processed = {item["name"] for item in folder_results if "name" in item}
    expected_names = {
        name
        for name in expected_names
        if not name.startswith("._")
        and Path(name).suffix.lower() in VALID_EXTENSIONS
        and name not in already_processed
    }
    drive_entries = {entry.name: entry for entry in gdrive.list_files(folder_path)}
    pending_entries = [
        drive_entries[name] for name in sorted(expected_names) if name in drive_entries
    ]

    missing_names = sorted(expected_names.difference(drive_entries))

    print("Pending images:", len(pending_entries))
    print("Missing images:", len(missing_names))

    for batch_index, batch in enumerate(chunks(pending_entries, DOWNLOAD_BATCH_SIZE)):
        batch_path = TEMP_PATH / f"batch_{batch_index:05d}"
        batch_path.mkdir(parents=True, exist_ok=True)
        try:
            downloaded = gdrive.download_entries(
                entries=batch,
                destination_path=batch_path,
                max_workers=DOWNLOAD_WORKERS,
            )
            for entry in tqdm(batch, desc=Path(folder_path).name, leave=False):
                local_path = downloaded.get(entry.name)
                if local_path is None:
                    print(f"Download failed: {entry.name}")
                    continue
                try:
                    start_time = time.perf_counter()
                    with torch.inference_mode():
                        prediction = detector.single_image_detection(str(local_path))

                    detections = prediction["detections"].xyxy
                    confidences = prediction["detections"].confidence
                    folder_results.append(
                        {
                            "name": entry.name,
                            "bounding_boxes": to_serializable(detections),
                            "accuracies": to_serializable(confidences),
                            "detection_time_sec": round(
                                time.perf_counter() - start_time, 4
                            ),
                        }
                    )
                except Exception as error:
                    print(f"Detection error in {entry.name}: {error}")
            save_json(OUTPUT_PATH, results)

        finally:
            shutil.rmtree(batch_path, ignore_errors=True)
    print(f"Stored detections: {len(folder_results)}")

save_json(OUTPUT_PATH, results)
print("\nFinished.")
print("Results:", OUTPUT_PATH)


Processing: Data/Sin clasificar/CAM01_02
Pending images: 370
Missing images: 0


CAM01_02:   0%|          | 0/16 [00:00<?, ?it/s]


0: 1280x1280 1 person, 256.5ms
Speed: 17.5ms preprocess, 256.5ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 1 person, 194.0ms
Speed: 13.5ms preprocess, 194.0ms inference, 4.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 2 persons, 197.5ms
Speed: 13.2ms preprocess, 197.5ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 2 persons, 196.1ms
Speed: 13.0ms preprocess, 196.1ms inference, 4.9ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 2 persons, 193.9ms
Speed: 13.3ms preprocess, 193.9ms inference, 0.4ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 3 persons, 176.2ms
Speed: 15.2ms preprocess, 176.2ms inference, 4.4ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 2 animals, 196.2ms
Speed: 13.5ms preprocess, 196.2ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 2 persons, 198.4ms
Speed: 13.5

CAM01_02:   0%|          | 0/16 [00:00<?, ?it/s]


0: 1280x1280 2 animals, 1 person, 245.5ms
Speed: 13.7ms preprocess, 245.5ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 196.7ms
Speed: 14.3ms preprocess, 196.7ms inference, 4.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 197.7ms
Speed: 13.8ms preprocess, 197.7ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 1 person, 197.7ms
Speed: 14.3ms preprocess, 197.7ms inference, 4.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 201.1ms
Speed: 16.5ms preprocess, 201.1ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 201.7ms
Speed: 18.8ms preprocess, 201.7ms inference, 7.6ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 198.5ms
Speed: 17.9ms preprocess, 198.5ms inference, 0.6ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 person, 201.4ms
Speed: 19.7ms pr

...


0: 1280x1280 (no detections), 252.4ms
Speed: 13.8ms preprocess, 252.4ms inference, 0.8ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 222.8ms
Speed: 13.9ms preprocess, 222.8ms inference, 1.1ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 222.9ms
Speed: 20.7ms preprocess, 222.9ms inference, 1.0ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 224.9ms
Speed: 14.9ms preprocess, 224.9ms inference, 1.1ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 224.8ms
Speed: 20.7ms preprocess, 224.8ms inference, 1.2ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 223.6ms
Speed: 20.5ms preprocess, 223.6ms inference, 1.3ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 222.6ms
Speed: 16.8ms preprocess, 222.6ms inference, 1.3ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 2 animals, 22

100EK113:   0%|          | 0/16 [00:00<?, ?it/s]


0: 1280x1280 1 animal, 258.9ms
Speed: 18.2ms preprocess, 258.9ms inference, 0.9ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 person, 227.6ms
Speed: 17.8ms preprocess, 227.6ms inference, 0.6ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 person, 227.3ms
Speed: 13.8ms preprocess, 227.3ms inference, 1.1ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 person, 229.7ms
Speed: 14.5ms preprocess, 229.7ms inference, 3.0ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 person, 228.6ms
Speed: 14.3ms preprocess, 228.6ms inference, 1.2ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 person, 232.6ms
Speed: 19.2ms preprocess, 232.6ms inference, 1.1ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 person, 231.4ms
Speed: 15.7ms preprocess, 231.4ms inference, 1.1ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 person, 234.4ms
Speed: 19.1ms preprocess, 234.4ms inf

100EK113:   0%|          | 0/10 [00:00<?, ?it/s]


0: 1280x1280 1 person, 255.1ms
Speed: 13.7ms preprocess, 255.1ms inference, 0.7ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 1 person, 226.4ms
Speed: 13.9ms preprocess, 226.4ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 231.2ms
Speed: 14.8ms preprocess, 231.2ms inference, 0.4ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 229.7ms
Speed: 15.7ms preprocess, 229.7ms inference, 0.6ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 227.5ms
Speed: 16.6ms preprocess, 227.5ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 person, 224.4ms
Speed: 16.3ms preprocess, 224.4ms inference, 0.7ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 person, 224.1ms
Speed: 22.1ms preprocess, 224.1ms inference, 1.2ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 person, 229.7ms
Speed: